# Practice: Does Nigerian-Language News Look Different?

**Computational Text Analysis Pack · Module 02 — Describing a Corpus**

*Technique:* A description worked end to end: length, richness, and what they license

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/02-describing-corpora/02f_practice_describing.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import math

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

The registered audience for this course is 81% Nigerian. So here is a question
with a real stake in it:

> **Is news written in the four Nigerian languages of this corpus descriptively
> different from news in the other twelve — in length, in vocabulary richness,
> or in both?**

A description module can answer that, but only if we are careful about the
difference between a real difference and an artefact of sampling.

### How to use this notebook

This is a worked case, not a tutorial. Run it once to see the shape of the
argument, then change the question at the top to one of your own and run it
again. The cells are written so that swapping the question is the only edit you
need.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# Step 1: state what we are comparing, and check the groups are comparable.
sub = df[df["n_words"] >= 30].copy()
sub["group"] = np.where(sub["is_nigerian"], "Nigerian languages", "Other African")
print(sub.groupby("group").agg(
    documents=("text", "size"),
    languages=("language", "nunique"),
    median_words=("n_words", "median"),
    mean_words=("n_words", "mean")).to_string(float_format=lambda v: f"{v:,.1f}"))
print()
print("Note the document counts differ. Every comparison below must use rates or")
print("medians, never raw totals — module 4's rule, applied before we start.")

In [ ]:
# Step 2: length. Median, not mean, because document length is skewed.
import numpy as np
a = sub.loc[sub["group"] == "Nigerian languages", "n_words"]
b = sub.loc[sub["group"] == "Other African", "n_words"]
print(f"median length  Nigerian {a.median():,.0f}   other {b.median():,.0f}")
print(f"IQR            Nigerian {a.quantile(.25):,.0f}-{a.quantile(.75):,.0f}   "
      f"other {b.quantile(.25):,.0f}-{b.quantile(.75):,.0f}")

# A permutation test, because a difference of medians on skewed data needs one.
rng = np.random.default_rng(0)
obs = a.median() - b.median()
pool = np.concatenate([a.values, b.values])
null = []
for _ in range(200):
    rng.shuffle(pool)
    null.append(np.median(pool[:len(a)]) - np.median(pool[len(a):]))
null = np.array(null)
print(f"\nobserved difference in medians: {obs:+,.0f} words")
print(f"under label shuffling: {null.min():+,.0f} to {null.max():+,.0f}")
print("CLEARS the null" if abs(obs) > np.abs(null).max() else "inside the null range")

plt.figure(figsize=(6.5, 3.4))
plt.hist(null, bins=20, alpha=.8, label="shuffled labels")
plt.axvline(obs, color="crimson", lw=2, label="observed")
plt.xlabel("difference in median length"); plt.legend(); plt.tight_layout()

In [ ]:
# Step 3: richness. MATTR, because the groups differ in length (module 02e).
def mattr(toks, window=200):
    if len(toks) <= window:
        return len(set(toks)) / max(len(toks), 1)
    return float(np.mean([len(set(toks[i:i+window])) / window
                          for i in range(0, len(toks) - window, window // 2)]))

samp = sub.groupby("group", group_keys=False).head(400).copy()
samp["MATTR"] = samp["text"].map(lambda t: mattr(tokenise(t)))
print(samp.groupby("group")["MATTR"].agg(["size", "mean", "std"]).to_string(
    float_format=lambda v: f"{v:.4f}"))
print()
print("Per language, since 'Nigerian' is four languages and not one thing:")
print(samp.groupby(["group", "language"])["MATTR"].mean()
      .sort_values(ascending=False).to_string(float_format=lambda v: f"{v:.4f}"))

## Step 4: what this licenses us to say

Write the finding as a sentence you would defend, then strike out what the data
does not support.

**Defensible:** "In this corpus, documents in Nigerian languages have a
different median length from those in other African languages, and the
difference survives label shuffling."

**Not defensible:** "Nigerian languages are more verbose." The corpus is *news*,
collected per outlet with a per-language cap this course applied. Length here is
a property of editorial practice at particular outlets, not of languages.

**The MATTR comparison is weaker still.** Look at the per-language table: the
spread *within* the Nigerian group is comparable to the gap between groups. A
group mean over four heterogeneous languages is close to meaningless, which is
the finding, and it is more interesting than the headline.

### 🔧 Your turn 1

Replace the Nigerian/other split with any two individual languages and re-run. Does the within-group spread still swamp the between-group difference?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Usually the two-language comparison is cleaner, because you have removed the
aggregation that was hiding heterogeneity. That is the general lesson of this
practice notebook: a grouping variable you did not construct (here, "is
Nigerian") is a hypothesis, not a fact about the data, and testing at the level
you actually care about is almost always more informative than testing the
convenient binary.

</details>